[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/two-phaseflow/JED116F/blob/main/notebooks/04_lumped_models_inverse.ipynb)

Click the badge to open this notebook in Google Colab. In Colab, choose **File > Save a copy in Drive** before you start, so your work is kept.

# JEÐ116F Notebook 04. Lumped models and inverse modelling

Lecture 9, Monday 2 November 2026. The parts follow the lecture slides, and each part names the slides it uses. The completed notebook is part of Assignment 4; the last section says what to hand in.

**Cell types**

- **A. Run and look.** The code is complete. Run it and read the output.
- **B. Fill the gaps.** There are seven gaps. Each `___` is one equation from the lecture slides, written in Python. The comment after the gap gives the slide and the equation. The test cell after it checks your line and explains the numbers it expects.
- **C. Extensions.** Open questions for when you finish early.

**Rules for this course**

1. Keep **units** next to every number in your code comments.
2. Nothing physical is typed in by hand when a property package can give it: water properties come from IAPWS-IF97 (`iapws`).
3. You may use AI tools for error messages, plotting and coding ideas. You must be able to explain every line you hand in.

**Parts**

1. One tank: the closed and the open tank (slides 15 to 18), and what storage and conductance do (slides 27 to 29)
2. More tanks: the balance of each tank stepped forward in time, and the ring model of notebook 02 as 250 tanks (slides 10 to 14, 19, 20)
3. Inverse modelling: fit a tank model to the 1997 production test at Hofsstaðir (slides 30 to 35)
4. The Hofsstaðir record 1997 to 2007: which model, which forecast, and what the parameters say about the reservoir (slides 37 to 50)

**Assignment 4** at the end: what to hand in.

**Symbols used throughout**

| symbol | meaning | unit |
|---|---|---|
| $W$ | production, mass flow out of the well | kg/s |
| $P,\ P_i$ | pressure; pressure in tank $i$ (the well is in tank 1) | Pa |
| $P_0$ | initial pressure, and the pressure outside an open model | Pa |
| $S_m$ | mass storativity | kg/(m² Pa) |
| $A,\ S_m A$ | area of a tank; storage of a tank | m², kg/Pa |
| $\sigma$ | conductance between two tanks, or between a tank and the outside | kg/(s Pa), or (kg/s)/bar |
| $w$ | mass flow between two tanks | kg/s |
| $\tau = S_m A/\sigma$ | time scale of an open tank | s |
| $T = kh/\nu$ | transmissivity | m s |
| $\rho,\ g$ | water density, gravitational acceleration | kg/m³, m/s² |
| $\phi,\ c_t,\ h$ | porosity, total compressibility, reservoir thickness | -, 1/Pa, m |
| RMS, $s$ | misfit, standard deviation of the data about the model | Pa, or m of water level |
| $n,\ p$ | number of data points, number of model parameters | |

## Part 0. Setup

Run both cells once at the start of every session.

In [ ]:
import os, urllib.request
import numpy as np                      # arrays and maths
import pandas as pd                     # reading the data files and printing tables
import matplotlib.pyplot as plt         # figures
from scipy.optimize import least_squares   # the search used in Part 3
from scipy.special import exp1          # the exponential integral, for the Theis curve in Part 2

# iapws gives the properties of water (IAPWS-IF97). Colab does not have it, so install it once per session.
try:
    from iapws import IAPWS97
except ImportError:
    !pip -q install iapws
    from iapws import IAPWS97

# The data files live in the course repository on GitHub. They are downloaded into a folder "data"
# next to the notebook, unless they are there already.
BASE = "https://raw.githubusercontent.com/two-phaseflow/JED116F/main/data/"
FILES = ["Hofsstadir_lumped_1997-2007.csv", "Ahuachapan_lumped_1968-2001.csv", "Gata_lumped_1982-2000.csv"]
os.makedirs("data", exist_ok=True)
for f in FILES:
    if not os.path.exists("data/" + f):
        urllib.request.urlretrieve(BASE + f, "data/" + f)

plt.rcParams.update({"font.size": 12, "axes.grid": True})

In [ ]:
# Units. Every calculation in this notebook is in SI units (kg, m, s, Pa).
# These names convert from the units we like to read: 3 * yr is three years in seconds,
# and a pressure in Pa divided by bar is the same pressure in bar.
minute, hour, day = 60.0, 3600.0, 86400.0     # s
yr = 365.25 * day                             # s
bar = 1e5                                     # Pa
g = 9.81                                      # gravitational acceleration [m/s2]
darcy = 9.869e-13                             # permeability unit [m2]


def water(P_MPa, T_C):
    '''Liquid water density rho [kg/m3] and kinematic viscosity nu [m2/s] from IAPWS-IF97,
    at pressure P_MPa [MPa] and temperature T_C [C].'''
    w = IAPWS97(P=P_MPa, T=T_C + 273.15)      # IAPWS97 wants MPa and kelvin
    return w.rho, w.mu / w.rho                # nu = mu / rho, dynamic viscosity over density


rho90, nu90 = water(2.0, 90.0)
print(f"water at 2 MPa and 90 C: rho = {rho90:.1f} kg/m3, nu = {nu90:.3e} m2/s")

## Part 1. One tank (slides 15 to 18)

A lumped model divides the reservoir into a few volumes, called tanks. Each tank has one pressure. The **storage** of a tank, $S_m A$, is the mass of water that must enter the tank to raise its pressure by 1 Pa (slide 12):

$$\Delta m = S_m A \, \Delta P$$

- $\Delta m$: mass added to the tank [kg]
- $S_m$: mass storativity, as in Lecture 6 [kg/(m² Pa)]
- $A$: area of the tank [m²]
- $\Delta P$: rise in pressure [Pa]

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_tank_closed.png" width="190"> &nbsp;&nbsp;&nbsp;&nbsp; <img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_tank_open.png" width="290">

**A closed tank** (left, slide 15) has no water coming in from outside. Every kilogram produced comes out of storage:

$$S_m A \, \frac{dP}{dt} = -W$$

With a constant production $W$ from $t = 0$ the pressure falls on a straight line (slide 16):

$$P = P_0 - \frac{W}{S_m A}\, t$$

- $P_0$: pressure before production started [Pa]
- $W$: production [kg/s]
- $t$: time since production started [s]

**An open tank** (right, slide 17) is joined through a conductance $\sigma$ to the outside, which stays at $P_0$. Water flows in at the rate $\sigma\,(P_0 - P)$, which grows as the tank pressure falls:

$$S_m A \, \frac{dP}{dt} = -W + \sigma\,(P_0 - P)$$

With a constant production from $t = 0$ the pressure falls and then levels off (slide 18):

$$P = P_0 - \frac{W}{\sigma}\left(1 - e^{-t/\tau}\right) \qquad \tau = \frac{S_m A}{\sigma}$$

- $\sigma$: conductance to the outside [kg/(s Pa)]
- $W/\sigma$: the final pressure drop, when the inflow equals the production [Pa]
- $\tau$: time scale [s]. At $t = \tau$ the drop is $1 - e^{-1} = 63$ % of the final drop.

### B1.1 The two solutions in code

Write the solutions of slides 16 and 18. `t` can be one time or an array of times; the same line works for both.

In [ ]:
def closed_tank(t, W, SA, P0):
    '''Pressure in one closed tank [Pa] at the times t [s], for a constant production W [kg/s]
    from t = 0. SA is the storage S_m A [kg/Pa]; P0 the pressure before production [Pa].'''
    return ___  # slide 16: P = P0 - W t / (S_m A)


def open_tank(t, W, SA, sigma, P0):
    '''Pressure in one open tank [Pa] at the times t [s], for a constant production W [kg/s]
    from t = 0. sigma is the conductance to the outside [kg/(s Pa)]; the outside stays at P0 [Pa].'''
    tau = SA / sigma  # slide 18: time scale tau = S_m A / sigma [s]
    return ___  # slide 18: P = P0 - (W/sigma) (1 - exp(-t/tau))

In [ ]:
# Tests, with the numbers of slides 16 and 18:
#   W = 20 kg/s, S_m A = 1e4 kg/Pa, sigma = 2e-4 kg/(s Pa), which is 20 (kg/s)/bar.
# We put P0 = 0, so the functions return the pressure change from the start (negative = a drop).
W0, SA0, sig0 = 20.0, 1e4, 2e-4

# Closed tank. Every second 20 kg leave a tank that takes 1e4 kg per Pa,
# so the pressure falls 20 / 1e4 = 2e-3 Pa every second.
# One year is 3.156e7 s, so after one year the drop is 2e-3 * 3.156e7 = 6.3e4 Pa = 0.63 bar.
drop_closed = -closed_tank(1 * yr, W0, SA0, 0.0)
print(f"closed tank: drop after one year {drop_closed / bar:.2f} bar (expected 0.63 bar)")
assert np.isclose(drop_closed, 0.631e5, rtol=0.01)

# Open tank, late time. After a long time the inflow sigma (P0 - P) equals the production W,
# so the drop stops at W / sigma = 20 / 2e-4 = 1e5 Pa = 1 bar.
drop_late = -open_tank(100 * yr, W0, SA0, sig0, 0.0)
print(f"open tank: drop after 100 years {drop_late / bar:.2f} bar (expected 1.00 bar)")
assert np.isclose(drop_late, 1e5, rtol=1e-6)

# Open tank at t = tau. tau = 1e4 / 2e-4 = 5e7 s = 1.6 years.
# At that time the drop is 1 - 1/e = 0.63 of the final drop, so 0.63 bar.
tau0 = SA0 / sig0
drop_tau = -open_tank(tau0, W0, SA0, sig0, 0.0)
print(f"open tank: tau = {tau0 / yr:.2f} yr, drop at t = tau {drop_tau / bar:.2f} bar (expected 0.63 bar)")
assert np.isclose(drop_tau, (1 - np.exp(-1)) * 1e5, rtol=1e-6)

# Open tank, very early. Almost no water has come in yet, so the open tank falls
# like the closed tank: after one day both have dropped by 2e-3 Pa/s * 86400 s = 173 Pa.
assert np.isclose(open_tank(day, W0, SA0, sig0, 0.0), closed_tank(day, W0, SA0, 0.0), rtol=0.01)
print("closed_tank and open_tank pass")

### A1.2 Closed and open, side by side (slide 9)

The same tank and production, once closed and once open. Both start on the same line. The open tank leaves it as water starts to flow in from outside, and levels off at $W/\sigma$.

In [ ]:
t = np.linspace(0, 5 * yr, 241)                      # 241 times from 0 to 5 years [s]
P_closed = closed_tank(t, W0, SA0, 0.0)              # pressure change in the closed tank [Pa]
P_open = open_tank(t, W0, SA0, sig0, 0.0)            # pressure change in the open tank [Pa]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t / yr, -P_closed / bar, label="closed tank: little or no recharge")
ax.plot(t / yr, -P_open / bar, label="open tank: recharge from outside")
ax.axhline(W0 / sig0 / bar, color="gray", ls=":", label=r"final drop of the open tank, $W/\sigma$")
ax.axvline(tau0 / yr, color="gray", ls="--", label=r"$\tau = S_m A/\sigma$")
ax.invert_yaxis()                                    # a pressure drop is plotted downward, like a falling water level
ax.set_xlabel("time since production started (years)"); ax.set_ylabel("pressure drop (bar)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### A1.3 What storage and conductance do (slides 27 to 29)

Change one parameter and keep the other. Left: three storages with the same conductance. Right: three conductances with the same storage, and $\sigma = 0$, which is the closed tank. The dashed lines on the left have the start slope $W/(S_m A)$; each reaches the final drop $W/\sigma$ at $t = \tau$.

| in the record | equation | depends on |
|---|---|---|
| start slope | $W/(S_m A)$ | storage |
| final drop | $W/\sigma$ | conductance |
| time to level off | $\tau = S_m A/\sigma$ | both |

Both parameters can be fitted only if the record contains both the start and the levelling off. A short test, the grey band on the right, contains only the start: from it we get the storage but not the conductance.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 6.2), sharey=True)
for f in [0.5, 1, 2]:                                 # half, the same, and double
    SA = f * SA0                                      # left: storage changes [kg/Pa], sigma stays sig0
    ax[0].plot(t / yr, -open_tank(t, W0, SA, sig0, 0.0) / bar,
               label=f"S_m A = {SA:.0e} kg/Pa, tau = {SA / sig0 / yr:.1f} yr")
    ax[0].plot([0, SA / sig0 / yr], [0, W0 / sig0 / bar], "k--", lw=0.8)   # start slope, reaches W/sigma at t = tau
    sig = f * sig0                                    # right: conductance changes [kg/(s Pa)], storage stays SA0
    ax[1].plot(t / yr, -open_tank(t, W0, SA0, sig, 0.0) / bar,
               label=f"sigma = {sig * bar:.0f} (kg/s)/bar, W/sigma = {W0 / sig / bar:.1f} bar")
ax[1].plot(t / yr, -closed_tank(t, W0, SA0, 0.0) / bar, "k:", label="sigma = 0 (closed)")
ax[1].axvspan(0, 0.3, color="0.9", zorder=0)          # a test of 0.3 years: the curves still lie on top of each other
ax[0].set_title("storage changes, sigma = 20 (kg/s)/bar")
ax[1].set_title("sigma changes, S_m A = 1e4 kg/Pa")
for a in ax:
    a.set_xlabel("time (years)")
    a.legend(fontsize=10, loc="upper center", bbox_to_anchor=(0.5, -0.14))   # legends below the plots, clear of the curves
ax[0].set_ylabel("pressure drop (bar)"); ax[0].set_ylim(3.4, -0.05)   # drop plotted downward
plt.show()

Optional: change both parameters with sliders (works in Colab and Jupyter).

In [ ]:
# Sliders for S_m A and sigma. ipywidgets is installed in Colab; elsewhere this cell may just print a message.
try:
    from ipywidgets import interact, FloatLogSlider
except ImportError:
    interact = None
    print("ipywidgets is not installed; skip this cell")

if interact is not None:
    def show_tank(SA=1e4, sigma=20.0):
        # one open tank at 20 kg/s; SA in kg/Pa, sigma in (kg/s)/bar (divided by bar to get kg/(s Pa))
        drop = -open_tank(t, W0, SA, sigma / bar, 0.0) / bar
        plt.figure(figsize=(9, 4))
        plt.plot(t / yr, drop)
        plt.axhline(W0 / sigma, color="gray", ls=":")            # the final drop W/sigma, in bar
        plt.ylim(7, -0.1); plt.xlabel("time (years)"); plt.ylabel("pressure drop (bar)")
        plt.title(f"tau = {SA / (sigma / bar) / yr:.2f} yr,   W/sigma = {W0 / sigma:.2f} bar")
        plt.show()

    interact(show_tank,
             SA=FloatLogSlider(value=1e4, base=10, min=3, max=5, step=0.05, description="S_m A"),
             sigma=FloatLogSlider(value=20, base=10, min=0.5, max=2, step=0.05, description="sigma"))

## Part 2. More tanks (slides 10 to 14, 19, 20)

In Lecture 6 the reservoir was cut into many thin rings around the well. A lumped model uses a few large ones, and draws them as a network: circles are storage, zigzags are conductances (slides 11 and 19).

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_rings_to_tanks.png" width="760">

**Flow between two tanks** (slide 13). Darcy's law between two tanks: water flows from the higher pressure to the lower one, at a rate proportional to the pressure difference.

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_sketch_flow.png" width="330">

$$w = \sigma\,(P_2 - P_1)$$

- $w$: mass flow from tank 2 into tank 1 [kg/s]
- $\sigma$: conductance between the tanks [kg/(s Pa)]; can be given in (kg/s)/bar
- $P_1, P_2$: pressures in the two tanks [Pa]

**Mass balance of a tank** (slide 14). What flows in minus what flows out changes the mass in the tank, and through the storage, its pressure:

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_sketch_balance.png" width="330">

$$S_m A_i \, \frac{dP_i}{dt} = w_i - w_{i-1}$$

- $S_m A_i$: storage of tank $i$ [kg/Pa]
- $w_i$: flow from tank $i+1$ into tank $i$, coming in from the far side [kg/s]
- $w_{i-1}$: flow from tank $i$ into tank $i-1$, going out toward the wells [kg/s]. For tank 1 this is the production $W$.

**Open model:** the last tank is joined through one more conductance to the outside at $P_0$. **Closed model:** it is not (slide 19).

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_tanks_three_open.png" width="560">

With two or more tanks the solution is not written out by hand. The computer steps the balance of every tank forward in time, as in notebook 02 for the rings.

### B2.1 Darcy between two tanks, and the balance of one tank

In [ ]:
def darcy_flow(sigma, P_far, P_near):
    '''Mass flow [kg/s] through a conductance sigma [kg/(s Pa)] from the tank at pressure P_far [Pa]
    into the tank at pressure P_near [Pa]. Positive when P_far is higher than P_near.'''
    return ___  # slide 13: w = sigma (P_2 - P_1)


def tank_rate(w_in, w_out, SA):
    '''Rate of pressure change dP/dt [Pa/s] of a tank with storage SA [kg/Pa],
    when the mass flow w_in [kg/s] comes in and w_out [kg/s] goes out.'''
    return ___  # slide 14: S_m A dP/dt = w_in - w_out

In [ ]:
# Tests with round numbers.

# Two tanks at 41 bar and 40 bar, joined by sigma = 1e-4 kg/(s Pa), which is 10 (kg/s)/bar.
# The pressure difference is 1 bar = 1e5 Pa, so 1e-4 * 1e5 = 10 kg/s flows
# from the tank at 41 bar into the tank at 40 bar.
w_test = darcy_flow(1e-4, 41 * bar, 40 * bar)
print(f"flow from 41 bar to 40 bar: {w_test:.1f} kg/s (expected 10 kg/s)")
assert np.isclose(w_test, 10.0)
# Swap the pressures and the flow goes the other way: -10 kg/s.
assert np.isclose(darcy_flow(1e-4, 40 * bar, 41 * bar), -10.0)

# A tank with storage 1e4 kg/Pa that receives 10 kg/s and loses 30 kg/s has a net loss of 20 kg/s,
# so its pressure falls at 20 / 1e4 = 2e-3 Pa/s. This is the closed tank of Part 1 again.
rate_test = tank_rate(10.0, 30.0, 1e4)
print(f"dP/dt with 10 kg/s in and 30 kg/s out: {rate_test:.1e} Pa/s (expected -2.0e-03 Pa/s)")
assert np.isclose(rate_test, -2e-3)
print("darcy_flow and tank_rate pass")

### A2.2 A chain of tanks

`tank_rates()` applies your two functions to every tank of a chain at once. Tank 1 holds the well. For each tank it finds the flow coming in from the far side (Darcy with the next tank, or with the outside at $P_0$ for the last tank of an open model) and the flow going out toward the well (Darcy with the previous tank, or the production $W$ for tank 1). Read the comments; you do not need to change this cell.

In [ ]:
def tank_rates(P, W, SA, sig, P0, is_open):
    '''dP/dt [Pa/s] of every tank in a chain of tanks.
    P        pressures of the tanks [Pa], tank 1 (with the well) first
    W        production from tank 1 [kg/s]
    SA       storage of each tank, S_m A_i [kg/Pa]
    sig      conductances [kg/(s Pa)]: sig[0] joins tanks 1 and 2, sig[1] joins tanks 2 and 3, and so on;
             in an open model the last one joins the last tank to the outside at P0
    P0       pressure outside [Pa], used only in an open model
    is_open  True for an open model, False for a closed one'''
    N = len(SA)                                  # number of tanks
    n_cond = N if is_open else N - 1             # number of conductances: an open model has one more, to the outside
    P_far = np.append(P[1:], P0)                 # for each tank, the pressure on its far side: the next tank, or P0 after the last
    w = np.zeros(N + 1)                          # w[i] = flow into tank i from tank i+1 (w[N] stays 0 in a closed model)
    w[0] = W                                     # the flow "out of tank 1 toward the well" is the production
    w[1:n_cond + 1] = darcy_flow(sig, P_far[:n_cond], P[:n_cond])   # Darcy through every conductance at once
    return tank_rate(w[1:], w[:-1], SA)          # each tank: in from its far side, out toward the well


# Tests.
P0_test = 50 * bar
# One closed tank producing 10 kg/s, storage 1e4 kg/Pa: falls at 10 / 1e4 = 1e-3 Pa/s.
assert np.isclose(tank_rates(np.array([P0_test]), 10.0, np.array([1e4]), np.array([]), P0_test, False)[0], -1e-3)
# One open tank 1 bar below the outside, no production: 1e-4 * 1e5 = 10 kg/s flows in,
# so its pressure rises at 10 / 1e4 = 1e-3 Pa/s.
assert np.isclose(tank_rates(np.array([P0_test - bar]), 0.0, np.array([1e4]), np.array([1e-4]), P0_test, True)[0], 1e-3)
# Two closed tanks 1 bar apart, no production: 10 kg/s flows from tank 2 into tank 1.
# Tank 1 (1e4 kg/Pa) rises at 1e-3 Pa/s; tank 2 (1e5 kg/Pa) falls at 1e-4 Pa/s. No mass is lost.
r = tank_rates(np.array([P0_test - bar, P0_test]), 0.0, np.array([1e4, 1e5]), np.array([1e-4]), P0_test, False)
assert np.allclose(r, [1e-3, -1e-4])
print("tank_rates passes")

### A2.3 Stepping forward in time

`simulate()` runs a chain of tanks through a production history and returns the pressure in tank 1, where the well is. You do not need to change this cell.

- The data files give the mean production over each interval between two measurements: `W[k]` is the mean between `t[k-1]` and `t[k]`.
- From the pressures now, `tank_rates()` gives how fast each pressure changes. A small step forward in time gives the new pressures, and so on. Each step uses the rates at the end of the step (an implicit step, as the solver of notebook 02 does), which stays stable for any step length.
- A step is at most one tenth of the shortest interval between measurements.

The test below runs one closed and one open tank and compares them with your solutions from B1.1. They should agree.

In [ ]:
def simulate(t, W, params, N, is_open, P0, nsub=None):
    '''Pressure in tank 1 [Pa] at the times t [s].
    params   [S_m A_1, ..., S_m A_N, sigma_1, ..., sigma_n]: N storages, then n conductances
             (n = N - 1 for a closed model, N for an open model)
    W[k]     mean production between t[k-1] and t[k] [kg/s]. All tanks start at P0.
    nsub     fixes the number of steps in every interval (used for log-spaced times)'''
    t = np.asarray(t, float); W = np.asarray(W, float)
    SA, sig = np.asarray(params[:N], float), np.asarray(params[N:], float)
    # The rates are a straight-line function of the pressures and the production:
    #     rates(P, W) = M @ P + c0 + d * W
    # Find M, c0 and d by calling tank_rates() with simple inputs, once at the start.
    zero = np.zeros(N)
    c0 = tank_rates(zero, 0.0, SA, sig, P0, is_open)                 # rates with all pressures 0 and no production
    d = tank_rates(zero, 1.0, SA, sig, P0, is_open) - c0              # extra rates per kg/s of production
    M = np.column_stack([tank_rates(np.eye(N)[j], 0.0, SA, sig, P0, is_open) - c0 for j in range(N)])   # per Pa in tank j
    P = np.full(N, float(P0))                                         # all tanks start at P0
    out = np.empty(len(t)); out[0] = P0
    inverses = {}
    h_max = np.min(np.diff(t)) / 10                                    # longest step: 1/10 of the shortest interval [s]
    for k in range(1, len(t)):
        dt = t[k] - t[k - 1]                                           # length of this interval [s]
        n_steps = nsub if nsub else max(10, int(np.ceil(dt / h_max - 1e-9)))
        h = dt / n_steps                                               # step length [s]
        key = round(h, 6)
        if key not in inverses:
            inverses[key] = np.linalg.inv(np.eye(N) - h * M)
        f = h * (c0 + d * W[k])
        for _ in range(n_steps):
            P = inverses[key] @ (P + f)      # implicit step: P_new = P_old + h * rates(P_new)
        out[k] = P[0]                        # keep the pressure in tank 1 at the end of the interval
    return out


# Test: the stepped solution against the exact solutions of B1.1, same tank and production as in Part 1.
Wc = np.full(len(t), W0)                                                  # constant 20 kg/s [kg/s]
err_closed = np.max(np.abs(simulate(t, Wc, [SA0], 1, False, 0.0) - closed_tank(t, W0, SA0, 0.0)))
err_open = np.max(np.abs(simulate(t, Wc, [SA0, sig0], 1, True, 0.0) - open_tank(t, W0, SA0, sig0, 0.0)))
print(f"largest difference from the exact solution: closed {err_closed:.1e} Pa, open {err_open / bar:.4f} bar")
assert err_closed < 1.0 and err_open < 0.005 * bar
print("simulate agrees with closed_tank and open_tank")

### A2.4 Two tanks: a small one around the well, a large one behind it (slide 19)

Tank 1 is small: it holds the well and reacts first. Tank 2 is ten times larger and lies behind it. At the start the pressure in tank 1 falls fast, as in a small closed tank. Then water from tank 2 flows in and the fall slows down. In the closed model it settles on a straight line set by the total storage, $W/(S_m A_1 + S_m A_2)$. In the open model it levels off, at a drop set by the two conductances in a row: $W\,(1/\sigma_1 + 1/\sigma_2)$.

In [ ]:
SA_1, SA_2 = 1e3, 1e4            # storage of tank 1 (around the well) and tank 2 (behind it) [kg/Pa]
sig_1, sig_2 = 4e-4, 2e-4        # conductance between tanks 1 and 2, and between tank 2 and the outside [kg/(s Pa)]

P_2c = simulate(t, Wc, [SA_1, SA_2, sig_1], 2, False, 0.0)            # two tanks, closed
P_2o = simulate(t, Wc, [SA_1, SA_2, sig_1, sig_2], 2, True, 0.0)      # two tanks, open

# Late time, closed: all the storage takes part, so the slope is W / (SA_1 + SA_2).
late = t > 3 * yr
slope_late = np.polyfit(t[late], P_2c[late], 1)[0]                     # slope of a straight line through the late part [Pa/s]
print(f"closed: late slope {slope_late * yr / bar:.3f} bar/yr, W/(SA_1 + SA_2) = {W0 / (SA_1 + SA_2) * yr / bar:.3f} bar/yr")
# Late time, open: the drop W/sigma_1 + W/sigma_2, the two conductances in a row.
print(f"open: drop after 5 years {-P_2o[-1] / bar:.2f} bar, W (1/sig_1 + 1/sig_2) = {W0 * (1 / sig_1 + 1 / sig_2) / bar:.2f} bar")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t / yr, -P_2c / bar, label="two tanks, closed")
ax.plot(t / yr, -P_2o / bar, label="two tanks, open")
ax.plot(t / yr, -closed_tank(t, W0, SA_1, 0.0) / bar, "k:", label="tank 1 alone, closed")
ax.set_ylim(4, -0.05); ax.set_xlabel("time (years)"); ax.set_ylabel("pressure drop in tank 1 (bar)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### A2.5 The ring model of notebook 02 is a tank model with 250 tanks (slides 10, 11 and 20)

In notebook 02 the reservoir around the well was cut into 250 rings. Each ring is a tank. Ring $i$ lies between an inner face at $r_{\mathrm{in}}$ and an outer face at $r_{\mathrm{out}}$, so its area is $\pi (r_{\mathrm{out}}^2 - r_{\mathrm{in}}^2)$ and its storage is

$$S_m A_i = S_m\,\pi\left(r_{\mathrm{out}}^2 - r_{\mathrm{in}}^2\right)$$

Between the centre of one ring, at $r_{\mathrm{this}}$, and the centre of the next ring out, at $r_{\mathrm{next}}$, the water flows radially. Darcy for radial flow, as in Lecture 6, gives the conductance

$$\sigma_i = \frac{2\pi T}{\ln\left(r_{\mathrm{next}}/r_{\mathrm{this}}\right)}$$

- $r_{\mathrm{in}}, r_{\mathrm{out}}$: inner and outer face of a ring [m]
- $r_{\mathrm{this}}, r_{\mathrm{next}}$: centres of two neighbouring rings [m]
- $T = kh/\nu$: transmissivity [m s]

The reservoir below has radius $R$ = 5 km and $kh$ = 10 darcy-m, and the well produces 20 kg/s. It is run twice: sealed at $R$ (closed), and held at $P_0$ at $R$ (open).

In [ ]:
kh = 10 * darcy                       # permeability-thickness [m3]
T = kh / nu90                         # transmissivity [m s], water at 90 C
phi, c_t, h = 0.1, 1e-9, 500.0        # porosity [-], total compressibility [1/Pa], thickness [m]
S_m = rho90 * phi * c_t * h           # mass storativity [kg/(m2 Pa)], storage by compression (slide 22)
R, r_w, n = 5000.0, 0.1, 250          # outer radius [m], well radius [m], number of rings

rf = np.geomspace(r_w, R, n + 1)      # the n + 1 ring faces, closer together near the well [m]
rc = np.sqrt(rf[:-1] * rf[1:])        # the n ring centres [m]

r_in, r_out = rf[:-1], rf[1:]         # inner and outer face of every ring [m]
SA_ring = S_m * np.pi * (r_out**2 - r_in**2)  # storage of every ring, S_m pi (r_out^2 - r_in^2) [kg/Pa]

r_this, r_next = rc[:-1], rc[1:]      # centre of every ring, and of the ring outside it [m]
sig_ring = 2 * np.pi * T / np.log(r_next / r_this)  # radial Darcy, 2 pi T / ln(r_next / r_this) [kg/(s Pa)]
sig_edge = 2 * np.pi * T / np.log(R / rc[-1])     # from the last ring centre to the edge at R [kg/(s Pa)]

In [ ]:
# Tests.
# The storages of all rings add up to the storage of the whole disc: S_m * pi * (R^2 - r_w^2).
print(f"total storage of the rings {SA_ring.sum():.3e} kg/Pa, of the disc {S_m * np.pi * (R**2 - r_w**2):.3e} kg/Pa")
assert np.isclose(SA_ring.sum(), S_m * np.pi * (R**2 - r_w**2))
# Conductances in a row add like resistances: the sum of 1/sigma from the first ring centre to R
# must equal 1/sigma of one conductance across the whole distance, ln(R/r_c1) / (2 pi T).
assert np.isclose(np.sum(1 / sig_ring) + 1 / sig_edge, np.log(R / rc[0]) / (2 * np.pi * T))
print("ring storages and conductances pass")

In [ ]:
# Run the 250-tank model for 30 years, with log-spaced times as in notebook 02.
t_r = np.r_[0, np.geomspace(minute, 30 * yr, 200)]       # times [s]
W_r = np.full(len(t_r), 20.0)                             # 20 kg/s [kg/s]
P_sealed = simulate(t_r, W_r, np.r_[SA_ring, sig_ring], n, False, 0.0, nsub=5)            # closed at R
P_held = simulate(t_r, W_r, np.r_[SA_ring, sig_ring, sig_edge], n, True, 0.0, nsub=5)     # held at P0 at R

# What one tank says about the late time:
slope_closed = -20.0 / (S_m * np.pi * R**2)               # one closed tank with all the storage: slope W/(S_m A) [Pa/s]
dP_open = 20.0 * np.log(R / rc[0]) / (2 * np.pi * T)      # one open tank: final drop W/sigma with sigma = 2 pi T / ln(R/r_c1) [Pa]
late = t_r > 20 * yr
assert np.isclose(np.polyfit(t_r[late], P_sealed[late], 1)[0], slope_closed, rtol=0.01)
assert np.isclose(-P_held[-1], dP_open, rtol=0.01)
theis = -20.0 / (4 * np.pi * T) * exp1(S_m * rc[0]**2 / (4 * T * t_r[1:]))      # Theis curve of Lecture 6 [Pa]

print(f"T = {T:.2e} m s, S_m = {S_m:.2e} kg/(m2 Pa); the pressure change reaches the edge after about "
      f"R^2 S_m / T = {R**2 * S_m / T / yr:.1f} years")
fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogx(t_r[1:] / day, -P_sealed[1:] / bar, label="rings, sealed at R")
ax.semilogx(t_r[1:] / day, -P_held[1:] / bar, label="rings, held at $P_0$ at R")
ax.semilogx(t_r[1:] / day, -theis / bar, "k--", label="Theis (Lecture 6)")
ax.axhline(dP_open / bar, color="C1", ls=":", label=r"open tank: $W/\sigma$")
ax.set_ylim(2.5 * dP_open / bar, 0)                        # drop plotted downward
ax.set_xlabel("time (days)"); ax.set_ylabel("drop in the first ring (bar)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

**The ring model against Theis** (slide 20). Early, the pressure change has not reached the edge, so the rings follow Theis and $T$ can be found from the slope against log time, as in a well test (Lecture 6). After about $R^2 S_m/(4T)$ the pressure change reaches the edge, and the curve bends away from the Theis line: the boundary effect of Lecture 6. With the edge sealed, the pressure then falls at $W/(S_m \pi R^2)$, as in one closed tank with all the storage. With the edge held at $P_0$, the pressure levels off at $W \ln(R/r_{c,1})/(2\pi T)$, as in one open tank. A lumped model describes this late part with a few tanks, for a production rate that changes every month.

## Part 3. Inverse modelling: the 1997 production test at Hofsstaðir (slides 30 to 35)

So far the parameters went in and the pressure came out: the **forward** problem, which is `simulate()`. With a record, we want the reverse: the parameters that reproduce the measured pressure. That is the **inverse** problem. There is no formula for it; we search (slide 30).

This part answers the first of the two questions on slide 30: how do we find the parameters from a record? We use a short record with a clear shape, the 140-day production test of 1997, and one or two tanks. Part 4 takes the second question: what can the fitted parameters forecast?

Hofsstaðir is a low-temperature field near Stykkishólmur, West Iceland, with water at about 87 C. Well HO-01 supplies the Stykkishólmur district heating. The data are the water level in HO-01, measured as depth below the wellhead, and the production of HO-01 (Guo, Axelsson, Ye and Xin, 2010, Proceedings World Geothermal Congress). The record starts with a production test on 19 March 1997.

Water level and pressure: when the level falls by $\Delta h$, the pressure at depth falls by

$$\Delta P = \rho\, g\, \Delta h$$

- $\Delta P$: pressure drop [Pa]
- $\rho$: water density [kg/m³]
- $g$: gravitational acceleration [m/s²]
- $\Delta h$: fall of the water level [m]

The production is given in L/s. One litre is 1/1000 m³, so $W = \rho\,Q/1000$ in kg/s, with $Q$ in L/s.

### A3.1 From the data file to pressure and mass production

In [ ]:
T_HOF_C = 87.0                                  # reservoir temperature at Hofsstadir (Guo et al. 2010) [C]
YEAR0 = 1997 + 77 / 365.25                      # 19.03.1997, the start of the record, as a decimal year
rho, nu_hof = water(1.0, T_HOF_C)               # density [kg/m3] and kinematic viscosity [m2/s] at 87 C

hof = pd.read_csv("data/Hofsstadir_lumped_1997-2007.csv", comment="#")   # lines starting with # are notes
t_all = hof["t_d"].values * day                 # time since 19.03.1997 [s]
Q_all = hof["Q_Ls"].values                      # production [L/s]
level = hof["level_m"].values                   # water level, depth below the wellhead [m]; larger is deeper
dh = level - level[0]                           # how far the level has fallen since the start [m]

W_all = rho * Q_all / 1000  # production in kg/s: W = rho Q / 1000
P_all = -rho * g * dh  # pressure change [Pa]: a fall dh is a drop rho g dh, so P - P0 = -rho g dh

In [ ]:
# Tests.
# On day 1 of the test HO-01 produced 20.47 L/s. At 968 kg/m3 that is 968 * 20.47 / 1000 = 19.8 kg/s.
print(f"density at 87 C: {rho:.1f} kg/m3")
print(f"production on day 1: {Q_all[1]:.2f} L/s = {W_all[1]:.2f} kg/s (expected 19.81 kg/s)")
assert np.isclose(W_all[1], 19.81, atol=0.01)
# The deepest level of the test was 29.3 m below the starting level:
# 968 kg/m3 * 9.81 m/s2 * 29.3 m = 2.78e5 Pa = 2.78 bar. Our pressures fall, so the change is -2.78 bar.
test = hof["t_d"].values <= 141                 # the 1997 production test: the first 141 days
print(f"largest pressure drop in the test: {-P_all[test].min() / bar:.2f} bar (expected 2.78 bar)")
assert np.isclose(-P_all[test].min() / bar, 2.78, atol=0.01)

t1, W1, P1 = t_all[test], W_all[test], P_all[test]    # the test: times [s], production [kg/s], pressure change [Pa]

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t1 / day, level[test], "k.", ms=4); ax[0].invert_yaxis(); ax[0].set_ylabel("water level (m)")
ax[1].step(t1 / day, W1, where="pre"); ax[1].set_ylabel("W (kg/s)"); ax[1].set_xlabel("days since 19.03.1997")
plt.show()

### B3.2 The misfit (slide 31)

How far is a model from the data? Take the difference at every measured point, square it so that positive and negative differences do not cancel, take the mean, and take the square root to get back to pressure units. This is the root mean square (RMS):

$$\mathrm{RMS} = \sqrt{\frac{1}{n}\sum_{j=1}^{n}\left(P_{\mathrm{model},j} - P_{\mathrm{obs},j}\right)^2}$$

- $n$: number of measured points
- $P_{\mathrm{model},j}$: model pressure at the time of point $j$ [Pa]
- $P_{\mathrm{obs},j}$: measured pressure at point $j$ [Pa]

In numpy, `np.mean(x)` is $\frac{1}{n}\sum x_j$ and `np.sqrt` is the square root. The best fit is the set of parameters with the smallest RMS.

In [ ]:
def rms(P_model, P_obs):
    '''Root mean square of the differences between model and data, in the units of the input.'''
    return ___  # slide 31: RMS = sqrt( mean( (P_model - P_obs)^2 ) )

In [ ]:
# Tests.
# A model that matches the data exactly has zero misfit.
assert np.isclose(rms(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])), 0.0)
# A model 1 unit above the data everywhere has a misfit of exactly 1.
assert np.isclose(rms(np.array([2.0, 3.0, 4.0]), np.array([1.0, 2.0, 3.0])), 1.0)
# Differences of +3 and -3: squares 9 and 9, mean 9, root 3. The signs do not cancel.
assert np.isclose(rms(np.array([3.0, -3.0]), np.array([0.0, 0.0])), 3.0)
print("rms passes")


def misfit(params, N, is_open, t, W, Pobs, P0=0.0, mask=None):
    '''RMS misfit [m of water level] of a tank model against measured pressures Pobs [Pa].
    The model is run with simulate(); mask picks the points that count (all measured points by default).
    Dividing by rho g turns the misfit from Pa into metres of water level, which is easier to read.'''
    if mask is None:
        mask = np.isfinite(Pobs)                 # skip times without a measurement
    Pm = simulate(t, W, params, N, is_open, P0)
    return rms(Pm[mask], Pobs[mask]) / (rho * g)

### A3.3 Three guesses with one open tank (slide 33)

One open tank has two parameters, $S_m A$ and $\sigma$. Each guess below gives one model curve and one misfit. The test lasted about 140 days, and the level fell about 30 m in the first 100 days at about 20 kg/s, so a good guess must have:

- a final drop $W/\sigma$ near 30 m of water, about 3 bar, so $\sigma$ near 20 / 3e5 = 7e-5 kg/(s Pa);
- a time scale $\tau = S_m A/\sigma$ shorter than the test, so that the level can level off within it.

The printout gives $W/\sigma$ and $\tau$ for each guess.

In [ ]:
guesses = [(1000, 3e-5), (200, 6e-5), (50, 1e-4)]       # (S_m A [kg/Pa], sigma [kg/(s Pa)])
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t1 / day, -P1 / (rho * g), "k.", ms=4, label="data")      # pressure change back to metres of water level
for SA, s in guesses:
    J = misfit([SA, s], 1, True, t1, W1, P1)                       # misfit of this guess [m]
    Pm = simulate(t1, W1, [SA, s], 1, True, 0.0)                    # model pressure change [Pa]
    print(f"S_m A = {SA:5d} kg/Pa, sigma = {s:.0e} kg/(s Pa):  W/sigma = {20 / s / (rho * g):5.1f} m of water, "
          f"tau = {SA / s / day:5.0f} days,  RMS = {J:.1f} m")
    ax.plot(t1 / day, -Pm / (rho * g), label=f"S_m A = {SA}, sigma = {s:.0e}: RMS {J:.1f} m")
ax.invert_yaxis(); ax.set_xlabel("days since 19.03.1997"); ax.set_ylabel("drawdown (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### A3.4 The misfit surface (slide 34)

Compute the misfit at every point of a grid of $S_m A$ and $\sigma$ values, and colour the grid by the misfit. Both axes are logarithmic, because storage and conductance can lie anywhere over several orders of magnitude. The lowest point is the best fit of the one-tank open model. This takes about 10 seconds.

In [ ]:
SA_grid = np.logspace(1, 4, 41)                 # 41 storages from 10 to 1e4 kg/Pa, evenly spaced in log10
sig_grid = np.logspace(-5.5, -3.5, 31)          # 31 conductances from 3e-6 to 3e-4 kg/(s Pa)
J = np.empty((len(sig_grid), len(SA_grid)))     # the misfit at every grid point [m]: one row per sigma, one column per S_m A
for i, s in enumerate(sig_grid):
    for j, SA in enumerate(SA_grid):
        J[i, j] = misfit([SA, s], 1, True, t1, W1, P1)

i, j = np.unravel_index(np.argmin(J), J.shape)  # row and column of the smallest misfit
best_grid = (SA_grid[j], sig_grid[i])
print(f"{J.size} model runs. Lowest misfit {J[i, j]:.2f} m at S_m A = {best_grid[0]:.0f} kg/Pa, "
      f"sigma = {best_grid[1]:.1e} kg/(s Pa) = {best_grid[1] * bar:.1f} (kg/s)/bar")

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(np.log10(SA_grid), np.log10(sig_grid), np.log10(J), 20, cmap="viridis")
plt.colorbar(cs, label=r"$\log_{10}$ RMS (m)")
ax.plot(np.log10(best_grid[0]), np.log10(best_grid[1]), "w+", ms=14, mew=2)   # the lowest grid point
ax.set_xlabel(r"$\log_{10}\, S_m A$ (kg/Pa)"); ax.set_ylabel(r"$\log_{10}\, \sigma$ (kg/(s Pa))")
plt.show()

### A3.5 A least-squares search (slide 35)

A grid needs many model runs, and with six parameters it would need far too many. `least_squares` starts from a guess, changes each parameter a little to find which way the misfit falls, steps that way, and repeats until no step lowers the misfit. It works on $\log_{10}$ of the parameters, for the same reason as the grid. The white line shows every point it tried.

In [ ]:
def fit(params0, N, is_open, t, W, Pobs, P0=0.0, mask=None, path=None):
    '''Search for the parameters with the smallest misfit, starting from params0.
    Works on log10 of the parameters. Returns the best parameters (not in log10).'''
    if mask is None:
        mask = np.isfinite(Pobs)
    lo = np.r_[np.full(N, -1.0), np.full(len(params0) - N, -14.0)]   # lower bounds on log10: storages, then conductances
    hi = np.r_[np.full(N, 13.0), np.full(len(params0) - N, 1.0)]     # upper bounds on log10
    def residuals(x):
        if path is not None:
            path.append(x.copy())                                   # remember every point tried, for the figure
        return simulate(t, W, 10**x, N, is_open, P0)[mask] - Pobs[mask]   # model minus data at the measured points [Pa]
    x0 = np.clip(np.log10(params0), lo + 1e-6, hi - 1e-6)
    r = least_squares(residuals, x0, bounds=(lo, hi), diff_step=1e-4)
    return 10**r.x


path = []
best_ls = fit([1000, 1e-5], 1, True, t1, W1, P1, path=path)          # start far from the minimum on purpose
path = np.array(path)
print(f"{len(path)} model runs. S_m A = {best_ls[0]:.0f} kg/Pa, sigma = {best_ls[1]:.2e} kg/(s Pa), "
      f"RMS = {misfit(best_ls, 1, True, t1, W1, P1):.2f} m")

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(np.log10(SA_grid), np.log10(sig_grid), np.log10(J), 20, cmap="viridis")
plt.colorbar(cs, label=r"$\log_{10}$ RMS (m)")
ax.plot(path[:, 0], path[:, 1], "w.-", ms=4, lw=0.8)
ax.set_xlim(1, 4); ax.set_ylim(-5.5, -3.5)
ax.set_xlabel(r"$\log_{10}\, S_m A$ (kg/Pa)"); ax.set_ylabel(r"$\log_{10}\, \sigma$ (kg/(s Pa))")
plt.show()

### A3.6 Test the method with a known answer

Make data from known parameters, add random noise, and fit. If the search works, it returns parameters close to the known ones. The noise here is 2.4 m of water level, the size of the misfit of the one-tank fit to the real data. It must be turned into a pressure with $\Delta P = \rho g \Delta h$ before it is added to the model pressures. The fit is repeated with 20 different noise draws, the Monte Carlo idea of Lecture 8.

In [ ]:
rng = np.random.default_rng(4)                  # random numbers, started from a fixed seed so the result repeats
true = np.array([200.0, 6e-5])                  # the known answer: S_m A [kg/Pa], sigma [kg/(s Pa)]
noise_m = 2.4                                   # size of the noise [m of water level]
noise_Pa = rho * g * noise_m  # the noise as a pressure [Pa]: Delta P = rho g Delta h

P_true = simulate(t1, W1, true, 1, True, 0.0)   # model pressures with the known parameters [Pa]
found = []
for k in range(20):
    P_syn = P_true + rng.normal(0, noise_Pa, len(P_true))       # the "data": model plus random noise [Pa]
    found.append(fit([1000, 1e-5], 1, True, t1, W1, P_syn))     # fit it, starting far away
found = np.array(found)

In [ ]:
# Tests.
# 2.4 m of water at 968 kg/m3 is 968 * 9.81 * 2.4 = 2.28e4 Pa, about 0.23 bar.
print(f"noise: {noise_m} m of water = {noise_Pa:.3e} Pa (expected 2.28e4 Pa)")
assert np.isclose(noise_Pa, 2.28e4, rtol=0.01)
# Every one of the 20 fits should land within 30 % of the known values.
assert np.all(np.abs(found / true - 1) < 0.3)
print("spread of the 20 fitted values (min to max):")
print(f"  S_m A: {found[:, 0].min():.0f} to {found[:, 0].max():.0f} kg/Pa   (true {true[0]:.0f})")
print(f"  sigma: {found[:, 1].min():.2e} to {found[:, 1].max():.2e} kg/(s Pa)   (true {true[1]:.0e})")

Noise of the same size as the real misfit moves the answer by a few percent only. So the search works, and the 2.4 m misfit of the real fit is not random noise: compare the fitted curve with the data in A3.5 or A3.7, where the differences follow a pattern. The one-tank model is too simple for this test.

### A3.7 More tanks on the same test (slide 37)

Fit models with more tanks. The six models of the ladder, with their numbers of parameters:

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_ladder_models.png" width="560">

Each model starts from the best fit of the model before it. A closed model becomes open by adding a conductance to the outside. An open model becomes closed with one more tank by turning the outside into a tank ten times larger than the last one. Starting each search from the previous fit keeps it close to the minimum.

In [ ]:
def model_name(N, is_open):
    '''A label such as "2 tank open".'''
    return f"{N} tank {'open' if is_open else 'closed'}"


def next_model(params, N, is_open, t_span):
    '''Starting guess for the next model on the ladder 1 closed, 1 open, 2 closed, 2 open, 3 closed, 3 open.'''
    SA, sig = params[:N], params[N:]
    if not is_open:                                   # closed -> open: add a weak conductance to the outside
        return np.r_[SA, sig, SA[-1] / (10 * t_span)], N, True
    return np.r_[SA, 10 * SA[-1], sig], N + 1, False  # open -> closed with one more tank, ten times larger


def closed1_start(t, W, Pobs, P0, mask):
    '''Best storage of one closed tank, found directly: the drop is (mass produced so far) / (S_m A).'''
    produced = np.r_[0, np.cumsum(W[1:] * np.diff(t))]   # mass produced since the start [kg]
    drop = P0 - Pobs                                       # pressure drop [Pa]
    return np.array([np.sum(produced[mask]**2) / np.sum(produced[mask] * drop[mask])])


def ladder(t, W, Pobs, P0=0.0, mask=None, models=6):
    '''Fit the models of the ladder in order. Returns {(N, is_open): best parameters}.'''
    if mask is None:
        mask = np.isfinite(Pobs)
    out = {}
    x, N, op = closed1_start(t, W, Pobs, P0, mask), 1, False
    for k in range(models):
        p = fit(x, N, op, t, W, Pobs, P0, mask)
        out[(N, op)] = p
        if k < models - 1:
            x, N, op = next_model(p, N, op, t[-1] - t[0])
    return out


test_fits = ladder(t1, W1, P1, models=4)        # the first four models: 1 closed, 1 open, 2 closed, 2 open
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t1 / day, -P1 / (rho * g), "k.", ms=4, label="data")
for (N, op), p in test_fits.items():
    J_ = misfit(p, N, op, t1, W1, P1)
    ax.plot(t1 / day, -simulate(t1, W1, p, N, op, 0.0) / (rho * g), label=f"{model_name(N, op)}: RMS {J_:.2f} m")
ax.invert_yaxis(); ax.set_xlabel("days since 19.03.1997"); ax.set_ylabel("drawdown (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

## Part 4. The Hofsstaðir record 1997 to 2007 (slides 37 to 50)

After the test, HO-01 stood idle until late 1999. Since then it has supplied the Stykkishólmur district heating, with more production in winter than in summer. The record ends on 21 April 2007; reinjection into well HO-02 started the next day, so the record has no reinjection (slide 45 shows how reinjection would enter a tank model).

This part takes the second question of slide 30: which model, and which forecast? Keep the rule of slide 29 in mind: a parameter can be fitted only if its effect is in the record. What is not yet in the record gives a range of forecasts (closed and open).

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(YEAR0 + t_all / yr, level, "k.", ms=3); ax[0].invert_yaxis(); ax[0].set_ylabel("water level (m)")
ax[1].step(YEAR0 + t_all / yr, W_all, where="pre"); ax[1].set_ylabel("W (kg/s)"); ax[1].set_xlabel("year")
x_inj = 2007 + 111 / 365.25                        # 22 April 2007: reinjection into HO-02 starts
for a in ax:
    a.axvline(x_inj, color="tab:orange", ls="--")
ax[0].text(x_inj - 0.1, 5, "reinjection into HO-02\nfrom 22 April 2007", ha="right", va="top", color="tab:orange")
plt.show()

### B4.1 The standard deviation (slide 39)

Each tank adds parameters, and the misfit can only fall when parameters are added. To compare models with different numbers of parameters, divide by the degrees of freedom $n - p$ instead of by $n$:

$$s = \sqrt{\frac{1}{n - p}\sum_{j=1}^{n}\left(P_{\mathrm{model},j} - P_{\mathrm{obs},j}\right)^2}$$

- $s$: standard deviation of the data about the model [Pa]
- $n$: number of measured points
- $p$: number of model parameters
- $P_{\mathrm{model},j} - P_{\mathrm{obs},j}$: the residual at point $j$ [Pa]

In numpy, `np.sum(x)` is $\sum x_j$.

In [ ]:
def std_dev(residuals, p):
    '''Standard deviation of the data about the model, from the residuals (model minus data, at every
    measured point) and the number of model parameters p. Same unit as the residuals.'''
    n = len(residuals)                                   # number of measured points
    return ___  # slide 39: s = sqrt( sum(residuals^2) / (n - p) )

In [ ]:
# Tests.
# Four residuals of +1, -1, +1, -1: the squares add up to 4.
# With p = 0 the standard deviation is sqrt(4 / 4) = 1, the same as the RMS.
res_test = np.array([1.0, -1.0, 1.0, -1.0])
assert np.isclose(std_dev(res_test, 0), 1.0)
# With p = 2 parameters only 4 - 2 = 2 degrees of freedom are left: sqrt(4 / 2) = 1.41.
# Each parameter is charged for, so a model with more parameters must fit better to win.
assert np.isclose(std_dev(res_test, 2), np.sqrt(2.0))
print("std_dev passes")

# The ladder on the whole record. This takes about half a minute.
hof_fits = ladder(t_all, W_all, P_all)
rows = []
for (N, op), p in hof_fits.items():
    res = simulate(t_all, W_all, p, N, op, 0.0) - P_all          # residuals at every measured point [Pa]
    rows.append(dict(model=model_name(N, op),
                     p=len(p),                                         # number of parameters
                     RMS_m=rms(res, 0.0) / (rho * g),                  # misfit [m of water level]
                     s_m=std_dev(res, len(p)) / (rho * g),             # standard deviation [m of water level]
                     sigma1_kgs_per_bar=p[N] * bar if len(p) > N else np.nan))   # first conductance [(kg/s)/bar]
table = pd.DataFrame(rows)
assert table["RMS_m"].iloc[3] < 2.5 and table["s_m"].iloc[3] > table["RMS_m"].iloc[3]
table.round(2)

The two-tank open and the three-tank closed models fit almost equally well (slide 41). In the three-tank open fit the conductance to the outside is close to zero, so it is the three-tank closed model again.

### A4.2 Forecast to 2038 (slide 42)

Continue production at 20 L/s until the end of 2038, the case that Guo et al. (2010) forecast, with the two models that fit best.

In [ ]:
Q_future = 20.0                                         # production after April 2007 [L/s]
W_future = rho * Q_future / 1000                        # the same in kg/s, as in A3.1
n_f = int((2039.0 - (YEAR0 + t_all[-1] / yr)) * yr / (10 * day))   # number of 10-day steps to the end of 2038
t_f = np.r_[t_all, t_all[-1] + np.arange(1, n_f + 1) * 10 * day]   # the record, then 10-day steps [s]
W_f = np.r_[W_all, np.full(n_f, W_future)]              # the measured production, then 20 L/s [kg/s]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(YEAR0 + t_all / yr, level, "k.", ms=3, label="data")
for key in [(2, True), (3, False)]:
    Pf = simulate(t_f, W_f, hof_fits[key], *key, 0.0)   # pressure change [Pa]
    lev = level[0] - Pf / (rho * g)                      # back to water level: a drop of rho g dh is dh metres deeper
    ax.plot(YEAR0 + t_f / yr, lev, label=f"{model_name(*key)}: {lev[-1]:.0f} m in 2038")
ax.invert_yaxis(); ax.set_xlabel("year"); ax.set_ylabel("water level (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.show()

### A4.3 Fit part of the record, predict the rest (slide 40)

A forecast can be tested on data not used in the fit. Fit only the data before spring 2003 (day 2200), predict 2003 to 2007, and compare the prediction with what was measured. `RMS_fit_m` is the misfit on the part used for the fit; `RMS_predict_m` is the misfit on the part not used in the fit.

In [ ]:
before = hof["t_d"].values < 2200                       # measurements used for the fit
after = ~before                                         # measurements kept back to test the prediction
fits_03 = ladder(t_all, W_all, P_all, mask=before)      # the ladder fitted to the data before 2003 only
rows = []
for (N, op), p in fits_03.items():
    Pm = simulate(t_all, W_all, p, N, op, 0.0)          # run the fitted model over the whole record [Pa]
    rows.append(dict(model=model_name(N, op), p=len(p),
                     RMS_fit_m=rms(Pm[before], P_all[before]) / (rho * g),       # misfit where it was fitted [m]
                     RMS_predict_m=rms(Pm[after], P_all[after]) / (rho * g)))    # misfit on the part not used in the fit [m]
val = pd.DataFrame(rows)
assert val["RMS_predict_m"].min() < 5
val.round(2)

### B4.4 What the storage and the conductance say about the reservoir (slides 46 to 50)

**Storage to area.** The fitted storage $S_m A$ gives an area only after a storage mechanism is chosen (slides 22 to 25):

$$A = \frac{S_m A}{S_m}, \qquad S_m = \rho\,\phi\, c_t\, h \quad \text{(compression)}, \qquad S_m = \frac{\phi}{g} \quad \text{(free surface)}$$

- $A$: area of the tank or the reservoir [m²]
- $S_m A$: fitted storage [kg/Pa]
- $\rho$: water density [kg/m³]; $\phi$: porosity [-]; $c_t$: total compressibility of rock and water [1/Pa]; $h$: reservoir thickness [m]; $g$: gravitational acceleration [m/s²]

Values for Hofsstaðir: $\phi$ = 0.1, $c_t = 10^{-9}$ 1/Pa, $h$ = 1000 m, $\rho$ of water at 87 C (`rho`, from IAPWS-IF97).

The water level in a well is a measure of the pressure at its feed zones, $\Delta P = \rho g \Delta h$, whether or not the reservoir has a free surface. Free-surface storage means that the water table in the rock itself moves. To decide, compare the areas with what is known about the field (slide 49): the hot ground (thermal gradient above 200 C/km in shallow holes) covers about 1.3 km², and the water level in the monitoring well AS-01, 1.2 km from HO-01, follows the level in HO-01.

**Conductance to permeability-thickness.** If the flow between tank 1 and tank 2 is radial, with tank 1 a disc of radius $R$ and tank 2 a ring from $R$ to $2R$, and the flow is taken between the middles of the two tanks, $r_1 = R/2$ and $r_2 = 3R/2$ (slide 50):

<img src="https://raw.githubusercontent.com/two-phaseflow/JED116F/main/figures/lumped_radial_tanks.png" width="300">

$$\sigma_1 = \frac{2\pi\, kh}{\nu \ln(r_2/r_1)} \qquad \text{so} \qquad kh = \frac{\sigma_1\, \nu \ln 3}{2\pi}$$

- $\sigma_1$: conductance between tanks 1 and 2 [kg/(s Pa)]
- $kh$: permeability-thickness [m³]
- $\nu$: kinematic viscosity of the water [m²/s], at 87 C (`nu_hof`)
- $\ln(r_2/r_1) = \ln 3$ whatever $R$ is

In [ ]:
phi, c_t, h_res = 0.1, 1e-9, 1000.0             # porosity [-], total compressibility [1/Pa], thickness [m]
SA_total = hof_fits[(3, False)][:3].sum()       # total storage of the 3-tank closed model [kg/Pa]
sig1 = hof_fits[(3, False)][3]                  # conductance between tanks 1 and 2 [kg/(s Pa)]

S_m_conf = ___  # slide 22: storage by compression, S_m = rho phi c_t h [kg/(m2 Pa)]
S_m_free = phi / g  # slide 23: storage by a free surface, S_m = phi / g [kg/(m2 Pa)]
A_conf = SA_total / S_m_conf  # area if the storage is compression, A = S_m A / S_m [m2]
A_free = SA_total / S_m_free  # area if the storage is a free surface, A = S_m A / S_m [m2]
kh_hof = sig1 * nu_hof * np.log(3) / (2 * np.pi)  # slide 50: kh = sigma_1 nu ln(3) / (2 pi) [m3]

In [ ]:
# Tests.
# Compression: 968 kg/m3 * 0.1 * 1e-9 1/Pa * 1000 m = 9.7e-5 kg/(m2 Pa).
# Free surface: 0.1 / 9.81 = 1.0e-2 kg/(m2 Pa), about 100 times more storage per m2.
print(f"S_m: compression {S_m_conf:.2e}, free surface {S_m_free:.2e} kg/(m2 Pa), ratio {S_m_free / S_m_conf:.0f}")
assert np.isclose(S_m_conf, 9.68e-5, rtol=0.01) and np.isclose(S_m_free, 1.02e-2, rtol=0.01)
# So the same fitted storage needs an area about 100 times larger if it is stored by compression.
print(f"total storage of the 3-tank closed model: {SA_total:.2e} kg/Pa")
print(f"area if compression:  {A_conf / 1e6:8.1f} km2")
print(f"area if free surface: {A_free / 1e6:8.2f} km2")
assert np.isclose(A_conf / A_free, S_m_free / S_m_conf)
# Tank 1 alone, and tanks 1 and 2, as discs (slides 48 and 49).
SA_1, SA_12 = hof_fits[(3, False)][0], hof_fits[(3, False)][:2].sum()
for name, S_m_ in (("compression", S_m_conf), ("free surface", S_m_free)):
    print(f"{name:13s}: tank 1 {SA_1 / S_m_ / 1e6:.3f} km2 (radius {np.sqrt(SA_1 / S_m_ / np.pi):.0f} m), "
          f"tanks 1 and 2 radius {np.sqrt(SA_12 / S_m_ / np.pi):.0f} m, all tanks radius {np.sqrt(SA_total / S_m_ / np.pi):.0f} m")
# kh: Guo et al. (2010) found about 5 darcy-m between the first and second tank with LUMPFIT.
print(f"sigma_1 = {sig1 * bar:.1f} (kg/s)/bar  gives  kh = {kh_hof / darcy:.1f} darcy-m")
assert 3 < kh_hof / darcy < 6

### Reflection

1. Which part of a record would be needed to choose between the open and the closed model at Hofsstaðir? Could more years of data settle it?
2. In A4.3, which model had the smallest misfit before 2003, and which had the smallest misfit after 2003? What does that tell you about adding tanks?
3. The storage gives an area only after you choose a storage mechanism. Which mechanism do you think applies at Hofsstaðir, and why? Use the areas from B4.4 and slide 49.
4. Guo et al. (2010, Fig. 11) forecast the same case with LUMPFIT, a 2-tank open and a 3-tank closed model, and got about 145 m and 285 m in 2038. Compare your two forecasts with theirs.
5. Name one thing that could make both forecasts wrong.

## Extensions (C)

**C1 Gata, initial level unknown.** The Gata file is from the Laugaland field in Holt, South Iceland, with water at about 100 C. It holds the water level in well GN-1 at Gata and the production at Laugaland, 1982 to 2000 (Axelsson, 1989). The file has no water level before week 65 (mid-1983), so the initial level must be fitted as well. Write a version of `fit()` that adds $P_0$ to the parameters (not in log10) and run the ladder. How well is $P_0$ determined?

**C2 Largest production.** Find the largest constant production that keeps the water level above 200 m until 2038, for the two-tank open and the three-tank closed models.

**C3 Fit the ring model.** Use `P_sealed` from A2.5 (sampled monthly for 10 years, with 0.05 bar noise) as data and fit tank models. Compare the total storage with $S_m \pi R^2$ and the sum of $1/\sigma_i$ with $\ln(R/r_{c,1})/(2\pi T)$. Which quantities does the fit recover, and which not?

**C4 Reinjection into tank 2.** Reinjection enters a tank model as an extra inflow to the tank it goes into (slide 45). Change `tank_rates` and `simulate` so that a reinjection rate $W_\mathrm{inj}$ enters tank 2. Run the 3-tank closed model with 20 L/s production from 2008 and 60 % of it reinjected into tank 2, and compare the water level in 2038 with the case without reinjection. Then put the same reinjection into tank 1 instead (net production $W - W_\mathrm{inj}$). Which case holds the level up more, and why?

In [ ]:
# C1 to C4: your code here

## Assignment 4: Ahuachapán

The file `Ahuachapan_lumped_1968-2001.csv` holds the reservoir pressure (bar gauge) and the net mass production (kg/s) of the Ahuachapán field, El Salvador, month by month from 1968. The pressure before production was 36.0 bar.

Hand in a short written report (PDF) with equations, numbers and figures, and your notebook. Define every symbol after each equation, with units.

1. Fit the six models of the ladder to the whole record. Give a table of $p$, RMS and $s$ in bar. How much does $s$ fall with each added tank, and where does adding a tank stop paying off?
2. Fit the 1-tank open, 2-tank closed and 2-tank open models to the data up to the end of 1985 only. Forecast the pressure to 2001 with the production that was measured, and compare with the measured pressure. Which model would you have trusted in 1985, and why?
3. Take the total storage of the 2-tank closed model fitted to the whole record. Compute the area it implies for confined liquid storage ($\phi$ = 0.1, $c_t$ = 1e-9 1/Pa, $h$ = 1000 m, water density at 230 C from `water()`) and for two-phase storage (take $c_t$ 1000 times larger). Compare both with the size of a geothermal field, a few to a few tens of km². Which explanation fits: two-phase storage, recharge from outside, or both?
4. Forecast the pressure in 2021 with the last production rate held constant, with the 2-tank closed and the 2-tank open models. In five to ten sentences, say which forecast you would give the field operator and one way the model could mislead them.